# 04. Accountability Frameworks

## 📚 Learning Objectives

By completing this notebook, you will:
- Apply accountability frameworks to AI systems
- Assign roles, governance, and redress
- Document and audit decision chains

## 🔗 Where this fits

**Builds on:** Course 05 (AIAT 115) — Unit 5, lesson 05 "Production Pipelines" — accountability needs a documented pipeline to attach names and dates to.

**Used later in:** Course 11 (AIAT 125) — Unit 5, lesson 06 "Model Versioning and Reproducibility" — the technical half of an audit trail.

---


# 04. Accountability Frameworks

## 🚨 THE PROBLEM: We Need Accountability for AI Decisions

**Remember the limitation from the previous notebook?**

We learned counterfactual analysis for "what if" explanations. But we discovered:

**How do we ensure accountability and responsibility for AI decisions?**

**The Problem**: Transparent AI systems also need:
- ❌ **Accountability frameworks** (who is responsible?)
- ❌ **Responsibility mechanisms** (how to assign responsibility?)
- ❌ **Audit trails** (how to track decisions?)
- ❌ **Stakeholder accountability** (who answers for outcomes?)

**We've learned:**
- ✅ How to use SHAP for explanations (Notebook 1)
- ✅ How to use LIME for fast explanations (Notebook 2)
- ✅ How to use counterfactuals for "what if" scenarios (Notebook 3)
- ✅ Multiple explanation methods

**But we haven't learned:**
- ❌ How to **define stakeholder responsibilities**
- ❌ How to **create audit trails**
- ❌ How to **establish responsibility mechanisms**
- ❌ How to **ensure accountability** for AI decisions

**We need accountability frameworks** to:
1. Define stakeholder responsibilities
2. Create audit trails
3. Establish responsibility mechanisms
4. Enable accountability for AI decisions

**This notebook solves that problem** by teaching you accountability frameworks for AI systems!

---

## 📚 Prerequisites (What You Need First)

**BEFORE starting this notebook**, you should have completed:
- ✅ **Example 1: SHAP Explanations** - Understanding explainability
- ✅ **Example 2: LIME Explanations** - Understanding local explanations
- ✅ **Example 3: Counterfactual Analysis** - Understanding "what if" scenarios
- ✅ **Basic Python knowledge**: Functions, data manipulation

**If you haven't completed these**, you might struggle with:
- Understanding why accountability matters
- Knowing how to structure accountability frameworks
- Understanding stakeholder responsibilities

---

## 🔗 Where This Notebook Fits

**This is the FOURTH example in Unit 4** - it teaches you accountability!

**Why this example FOURTH?**
- **Before** you can ensure accountability, you need explainability (Examples 1-3)
- **Before** you can implement HITL, you need accountability structures
- **Before** you can build transparent systems, you need accountability

**Builds on**: 
- 📓 Example 1: SHAP Explanations (explainability)
- 📓 Example 2: LIME Explanations (local explanations)
- 📓 Example 3: Counterfactual Analysis ("what if" scenarios)

**Leads to**: 
- 📓 Example 5: Human-in-the-Loop (HITL approaches)
- 📓 Example 6: Transparency Tools (transparency frameworks)

**Why this order?**
1. Accountability provides **responsibility structures** (needed for ethical AI)
2. Accountability teaches **stakeholder roles** (critical for governance)
3. Accountability shows **audit mechanisms** (essential for transparency)

---

## The Story: Who Is Responsible?

Imagine you're using an AI system that makes a wrong decision. **Before** accountability frameworks, you wouldn't know who to hold responsible (developers? data scientists? users?). **After** implementing accountability frameworks, you have clear responsibilities, audit trails, and accountability mechanisms!

Same with AI: **Before** we have explanations but no accountability, now we learn accountability frameworks - define responsibilities, create audit trails, establish accountability! **After** accountability frameworks, we have responsible and accountable AI systems!

---

## Why Accountability Frameworks Matter

Accountability frameworks are essential for ethical AI:
- **Responsibility**: Define who is responsible for AI decisions
- **Transparency**: Enable tracking and auditing of decisions
- **Trust**: Build user confidence through accountability
- **Compliance**: Meet regulatory requirements for accountability
- **Ethics**: Ensure responsible AI development and deployment

## Learning Objectives
1. Understand accountability frameworks
2. Learn stakeholder responsibilities
3. Create audit trails
4. Establish responsibility mechanisms
5. Implement model cards and data sheets
6. Build accountability structures

## 📌 The case: an algorithm that brought down a government

**The Dutch childcare benefits scandal (*toeslagenaffaire*).**

Between roughly 2005 and 2019 the Netherlands' Tax and Customs Administration
(*Belastingdienst*) ran risk models to flag childcare-benefit claims for fraud investigation.
Roughly **26,000 families** were wrongly accused and ordered to repay their allowances in
full — typically **€20,000 to €60,000** each. Families were driven into debt, bankruptcy and
family breakdown; children were taken into care.

The accountability trail is worth reading as a sequence:

- **15 January 2021** — the entire Rutte III cabinet resigns over the affair. A whole
  government, over a risk model.
- **7 December 2021** — the Dutch Data Protection Authority fines the Minister of Finance
  **€2.75 million** for processing applicants' **(dual) nationality** in a manner it found
  unlawful, discriminatory and improper. The regulator described the breach as severe enough
  that it set aside its usual fine-calculation policy.

Now the part that matters for this notebook. At no point in fifteen years was there a person
whose job it was to answer for the model's decisions. There were procedures. There were
lawyers. There was no **name** attached to "this system's outputs are correct and
proportionate", and no trail that let a family reconstruct why they had been flagged.

**The WHY — what goes wrong without accountability structures.** Explanations (Notebooks
01–03) tell you *why one decision came out that way*. They do not tell you **who is
answerable** when it is wrong, **what evidence exists** six months later when it is
challenged, or **how the affected person gets it undone**. Those are three concrete
artefacts, and you are about to build all three: a RACI matrix, a real audit trail over 223
real decisions with the SHAP factor stored for each, and a redress path with a named owner.

---


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- The **real Titanic passenger manifest** (`Course 04/datasets/raw/titanic.csv`) and the
  same survival model explained in Notebooks 01-03, used here as the high-stakes
  screening system whose decisions must be accounted for.
- `pandas`, `scikit-learn`, `shap`, `hashlib` - the audit trail stores the real
  explanation SHAP computed for each decision.

**Outputs:** What you'll see when you run the cells

- A RACI responsibility matrix for the system's lifecycle.
- A real audit trail: every test-set decision logged with its input hash,
  predicted probability and top explanatory factor.
- The audit queries a regulator would actually run - including a **measured**
  count of decisions that should have gone to a human and did not.

---


## Part 1: Accountability as Data Structures

Accountability means three concrete things you can build:
1. A **responsibility matrix** (who is Responsible / Accountable / Consulted / Informed
   for each stage - "RACI")
2. An **audit trail** (every significant decision logged with enough context to reconstruct it)
3. A **redress path** (an appeal route with a named owner)

We build all three around the **real** model from Notebooks 01-03 - the classifier
trained on the Titanic manifest, standing in for any system that screens people.
The audit trail below holds real predictions about real passengers, which is what
makes the audit queries at the end return real numbers instead of a story.


In [1]:
# Why RACI: accountability fails when it is nobody's job - this matrix forces
# every lifecycle stage to name the human who answers for it.

# Step 1: A RACI responsibility matrix for the screening system of Notebooks 01-03
import pandas as pd
from datetime import datetime, timezone

# One row per lifecycle stage; the columns assign who does the work (R),
# who owns the outcome (A), who advises (C), and who must be told (I).
raci = pd.DataFrame([
    ['Data collection',      'Data Engineer',   'Head of Data',    'Legal/DPO',    'Operations'],
    ['Model training',       'ML Engineer',     'ML Lead',         'Ethics Board', 'Operations'],
    ['Fairness evaluation',  'ML Engineer',     'Ethics Board',    'Legal/DPO',    'Executives'],
    ['Deployment decision',  'ML Lead',         'Product Owner',   'Ethics Board', 'All staff'],
    ['Individual decisions', 'The AI system',   'Case Officer',    'ML Lead',      'Affected person'],
    ['Appeals / redress',    'Case Officer',    'Head of Operations', 'Legal/DPO', 'Affected person'],
], columns=['Stage', 'Responsible', 'Accountable', 'Consulted', 'Informed'])

print("RACI RESPONSIBILITY MATRIX - the screening model from Notebooks 01-03")
print("=" * 95)
print(raci.to_string(index=False))
print("\nKey rule: 'Accountable' is always a PERSON or a named role - never")
print("'the algorithm'. Row 5 makes that explicit: the system is responsible for")
print("producing the decision, but a human (Case Officer) is accountable for it,")
print("and the person the decision is about must be informed of it.")


RACI RESPONSIBILITY MATRIX - the screening model from Notebooks 01-03
               Stage   Responsible        Accountable    Consulted        Informed
     Data collection Data Engineer       Head of Data    Legal/DPO      Operations
      Model training   ML Engineer            ML Lead Ethics Board      Operations
 Fairness evaluation   ML Engineer       Ethics Board    Legal/DPO      Executives
 Deployment decision       ML Lead      Product Owner Ethics Board       All staff
Individual decisions The AI system       Case Officer      ML Lead Affected person
   Appeals / redress  Case Officer Head of Operations    Legal/DPO Affected person

Key rule: 'Accountable' is always a PERSON or a named role - never
'the algorithm'. Row 5 makes that explicit: the system is responsible for
producing the decision, but a human (Case Officer) is accountable for it,
and the person the decision is about must be informed of it.


In [2]:
# Why audit trails: when a decision is challenged months later, the trail is
# the only way to reconstruct WHO/WHAT/WHEN - no trail, no accountability.
# Why on real decisions: a trail of invented records proves nothing. Below we log
# every decision the REAL model makes about REAL passengers, then run the queries
# an auditor would run and report whatever they return.

# Step 2: An audit trail over the real model's decisions
import hashlib, json
import numpy as np
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# --- The real system: same data and model as Notebooks 01-03 ---
df = pd.read_csv('../../../Course 04/datasets/raw/titanic.csv')
df['Age'] = df['Age'].fillna(df['Age'].median())          # 177 real missing ages
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
X = pd.DataFrame({
    'Pclass':    df['Pclass'].astype(float),
    'Age':       df['Age'].astype(float),
    'SibSp':     df['SibSp'].astype(float),
    'Parch':     df['Parch'].astype(float),
    'Fare':      df['Fare'].astype(float),
    'is_female': (df['Sex'] == 'female').astype(float),
})
y = df['Survived'].values
group = df['Sex'].values   # kept alongside so the audit can ask WHO was affected
X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, group, test_size=0.25, random_state=42, stratify=y)
MODEL_VERSION = 'screening-rf-v1.0'
model = RandomForestClassifier(n_estimators=150, random_state=42).fit(X_train, y_train)
print(f"System under audit: {MODEL_VERSION} trained on {len(X_train)} real passengers")
print(f"Test accuracy on {len(X_test)} held-out passengers: "
      f"{accuracy_score(y_test, model.predict(X_test)):.3f}")

# Real explanations to store WITH each decision: SHAP tells us which feature drove
# this particular prediction, so the record can be defended months later.
sv = shap.TreeExplainer(model).shap_values(X_test)
sv1 = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if sv.ndim == 3 else sv)

# --- The policy, and what the system was actually configured to do ---
POLICY_THRESHOLD = 0.70    # policy: a human must review any decision below this confidence
DEPLOYED_THRESHOLD = 0.60  # what the deployed router was actually set to (a real-world gap)

audit_log = []

def log_decision(model_version, features, prediction, confidence, explanation,
                 human_reviewer=None, subject_group=None, actual_outcome=None):
    """Append one reconstructable decision record to the audit trail.
    The input is stored as a HASH: enough to prove which input produced this
    decision, without copying personal data into a second place (Unit 3!)."""
    record = {
        'timestamp': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'model_version': model_version,
        'input_hash': hashlib.sha256(
            json.dumps(features, sort_keys=True).encode()).hexdigest()[:12],
        'prediction': prediction,
        'confidence': float(confidence),
        'top_factors': explanation,
        'human_reviewer': human_reviewer,   # None = fully automated
        'subject_group': subject_group,
        'actual_outcome': actual_outcome,
    }
    audit_log.append(record)
    return record

# --- Log every decision the system made on the held-out passengers ---
proba = model.predict_proba(X_test)
confidence = proba.max(axis=1)
pred = model.predict(X_test)
for i in range(len(X_test)):
    contrib = pd.Series(sv1[i], index=X_test.columns)
    top2 = contrib.sort_values(key=abs, ascending=False).head(2)
    log_decision(
        MODEL_VERSION,
        X_test.iloc[i].to_dict(),
        'predicted survivor' if pred[i] == 1 else 'predicted non-survivor',
        confidence[i],
        [f"{f} {'+' if v > 0 else '-'}" for f, v in top2.items()],
        # The deployed router - not the policy - decided who saw a human.
        human_reviewer=('case.officer@org' if confidence[i] < DEPLOYED_THRESHOLD else None),
        subject_group=g_test[i],
        actual_outcome=int(y_test[i]),
    )

print(f"\nAUDIT TRAIL - {len(audit_log)} decisions logged")
print("=" * 100)
for r in audit_log[:6]:
    print(f"  {r['timestamp']} | {r['model_version']} | input {r['input_hash']} | "
          f"{r['prediction']:<22} | conf {r['confidence']:.2f} | "
          f"reviewer: {r['human_reviewer'] or 'AUTOMATED':<16} | "
          f"factors: {', '.join(r['top_factors'])}")
print(f"  ... {len(audit_log) - 6} further records")

# --- The queries an auditor actually runs, answered from the trail ---
log_df = pd.DataFrame(audit_log)
automated = log_df['human_reviewer'].isna()
violations = log_df[automated & (log_df['confidence'] < POLICY_THRESHOLD)]
print("\n" + "=" * 100)
print("AUDIT QUERIES")
print("=" * 100)
print(f"1. Decisions made without human review: {int(automated.sum())} of {len(log_df)} "
      f"({automated.mean():.1%})")
print(f"2. Policy says a human reviews anything below confidence "
      f"{POLICY_THRESHOLD:.2f}.")
print(f"   Automated decisions below that threshold: {len(violations)} "
      f"({len(violations)/len(log_df):.1%} of all decisions)")
if len(violations):
    print(f"   -> The deployed router was set to {DEPLOYED_THRESHOLD:.2f}, not "
          f"{POLICY_THRESHOLD:.2f}. The trail turned a")
    print("      configuration gap into a countable finding, which is the whole point.")
    print("\n   Who did those decisions affect?")
    print(violations.groupby('subject_group').size().to_string())
    agreement = (violations['actual_outcome'] ==
                 (violations['prediction'] == 'predicted survivor').astype(int)).mean()
    print(f"   Accuracy on those unreviewed low-confidence decisions: {agreement:.1%} "
          f"(vs {accuracy_score(y_test, pred):.1%} overall)")
    print("   Exactly the cases policy wanted a human to see - and the model is")
    print("   measurably weaker on them.")

# --- Redress: reconstruct one challenged decision from the trail alone ---
challenged = violations.iloc[0] if len(violations) else log_df.iloc[0]
print("\n" + "=" * 100)
print("REDRESS: reconstructing one challenged decision from the trail")
print("=" * 100)
for k, v in challenged.items():
    print(f"  {k:>15}: {v}")
print("\nThe input hash lets the Case Officer prove WHICH record produced this")
print("decision without storing the personal data twice; the stored factors say")
print("WHY; the model version says WHICH system. That is what makes an appeal")
print("answerable instead of a matter of opinion.")


System under audit: screening-rf-v1.0 trained on 668 real passengers
Test accuracy on 223 held-out passengers: 0.789



AUDIT TRAIL - 223 decisions logged
  2026-08-25T18:38:03+00:00 | screening-rf-v1.0 | input b95ad9306764 | predicted non-survivor | conf 0.92 | reviewer: AUTOMATED        | factors: is_female -, Fare -
  2026-08-25T18:38:03+00:00 | screening-rf-v1.0 | input 82cc2375c913 | predicted survivor     | conf 0.95 | reviewer: AUTOMATED        | factors: is_female +, Fare +
  2026-08-25T18:38:03+00:00 | screening-rf-v1.0 | input 5c715999fcb1 | predicted non-survivor | conf 0.81 | reviewer: AUTOMATED        | factors: is_female -, Pclass -
  2026-08-25T18:38:03+00:00 | screening-rf-v1.0 | input ca038c243ed6 | predicted survivor     | conf 0.83 | reviewer: AUTOMATED        | factors: is_female +, Fare +
  2026-08-25T18:38:03+00:00 | screening-rf-v1.0 | input fe19ea4112f5 | predicted non-survivor | conf 0.97 | reviewer: AUTOMATED        | factors: Pclass -, is_female -
  2026-08-25T18:38:03+00:00 | screening-rf-v1.0 | input 2ca792b0f0c3 | predicted survivor     | conf 0.93 | reviewer: AUTOMATED   

## 💬 Discuss

You have just logged **223 real decisions** with an input hash, a predicted probability, the
top explanatory factor and a reviewer field — and then run the queries a regulator would run,
including a **measured** count of decisions that should have gone to a human and did not.

1. **Who signs row 5?** In your RACI matrix the *Responsible* party for individual decisions
   is "The AI system" and the *Accountable* party is a named Case Officer. Is that honest, or
   is it a way of assigning blame to the person with the least power in the process? Whose
   name would you put in the *Accountable* column for a system that makes 10,000 decisions a
   day — and can any single human meaningfully accept accountability at that volume? If not,
   what is the alternative that is not "nobody"?
2. **Your audit trail stores an explanation. Is that a liability?** Every logged decision now
   carries the factors that drove it, indefinitely. That is exactly what a wronged family in
   the Dutch case never had. It is also a permanent, discoverable record of every mistake your
   organisation has made, plus personal data that Unit 3 says you must secure and may have to
   erase. How long do you keep it? Who can read it? What do you do when Article 17 erasure and
   the audit obligation point in opposite directions?
3. **What makes redress real?** Write the appeal path for this system: who receives the
   appeal, what they are empowered to change, what deadline binds them, and what happens if
   they do nothing. Then check it against the Dutch case — would your path have surfaced
   26,000 wrongful accusations in year one, or in year fifteen? Be honest about which
   component of your design does that work.


---

## 🚫 When Accountability Frameworks Hit a Limitation

### The Limitation We Discovered

We've learned accountability frameworks for defining responsibilities. **But there's still a challenge:**

**How do we incorporate human judgment into AI decision-making?**

Accountability frameworks work well when:
- ✅ We have clear responsibilities defined
- ✅ We have audit trails in place
- ✅ We have accountability mechanisms

**But ethical AI systems also need:**
- ❌ **Human oversight** (human judgment for critical decisions)
- ❌ **Human-in-the-loop** (HITL) approaches
- ❌ **Human review** for uncertain cases
- ❌ **Human validation** of AI decisions

### Why This Is a Problem

When we have accountability but no human oversight:
- Critical decisions may be made without human judgment
- Uncertain cases may not get human review
- AI decisions may lack human validation
- We may miss important context that humans understand

### The Solution: Human-in-the-Loop (HITL) Approaches

We need **human-in-the-loop approaches** to:
1. Incorporate human judgment into AI decisions
2. Enable human review for uncertain cases
3. Provide human oversight for critical decisions
4. Combine AI efficiency with human judgment

**This is exactly what we'll learn in the next notebook: Human-in-the-Loop Approaches!**

---

## ➡️ Next Steps

**You've completed this notebook!** Now you understand:
- ✅ How to use SHAP, LIME, and counterfactuals (Notebooks 1-3)
- ✅ How to establish accountability frameworks (This notebook!)
- ✅ **The limitation**: We need human oversight!

**Next notebook**: `05_hitl_approaches.ipynb`
- Learn about human-in-the-loop approaches
- Understand human oversight mechanisms
- Implement HITL for critical decisions
- Combine AI with human judgment

## ⚠️ Where this breaks

- **A RACI matrix is a claim, not a control.** Writing "Accountable: Ethics Board" changes
  nothing unless that board has the authority to stop a launch and has visibly used it. The
  Dutch tax administration had an organisation chart. Ask instead: **when did this body last
  say no, and what happened?**
- **Audit logs prove what the system did, not whether it should have.** Your trail records
  probabilities and top factors beautifully and contains no evidence about whether the model
  was appropriate for the purpose, whether the training data was lawfully obtained, or
  whether the affected person understood what was happening.
- **The log is itself personal data with its own risks.** It holds hashed inputs, predictions
  and explanations about identifiable people, so it inherits every Unit 3 obligation:
  retention limits, access control, breach exposure, subject access. Teams routinely build
  audit trails with no retention policy at all, which converts a compliance control into a
  compliance liability.
- **Accountability at volume needs sampling and statistics, not signatures.** Nobody reviews
  10,000 decisions a day. The realistic design is a named owner plus **continuous monitoring**
  plus **random audit of a sample**, with an escalation threshold — and the sampling rate is
  the real policy decision, because it sets how long a systematic error runs before you see it.
- **A trail nobody queries is a filing cabinet.** The value came from the *queries* you ran
  above, not from the logging. If no one runs them on a schedule, and no one is accountable
  for acting on what they return, the artefact is theatre.

**The cheaper alternative when a full framework is disproportionate:** for a low-risk internal
tool, a **one-page decision record** — what the system does, who owns it, how someone
complains, when it will be reviewed — delivers most of the accountability at a fraction of the
cost. Escalate to the full RACI, trail and redress path when the decision affects someone's
liberty, money, health or employment; the Dutch case is what the bottom of that scale looks
like when you get it wrong.


## 📚 References

1. Mitchell, M., Wu, S., Zaldivar, A., et al. (2019). *Model Cards for Model Reporting*. FAT* 2019. <https://arxiv.org/abs/1810.03993>
2. Gebru, T., Morgenstern, J., Vecchione, B., et al. (2021). *Datasheets for Datasets*. Communications of the ACM, 64(12). <https://arxiv.org/abs/1803.09010>
3. Raji, I. D., Smart, A., White, R. N., et al. (2020). *Closing the AI Accountability Gap: Defining an End-to-End Framework for Internal Algorithmic Auditing*. FAT* 2020. <https://arxiv.org/abs/2001.00973>